# Dataset genereren óp Kaggle (PDOK + BAG)

Dit notebook draait de datageneratie-pipeline volledig op Kaggle, zodat je niets
vanaf je eigen machine hoeft te uploaden. De dataset verschijnt als *notebook output*
en koppel je daarna als *Input* aan het trainingsnotebook (`kaggle_train_yolo.ipynb`).

**Vereisten:**
1. *Settings → Internet* op **On** (vraagt eenmalig telefoonverificatie van je Kaggle-account).
2. Accelerator: **CPU volstaat** — bewaar je GPU-quotum voor de training.
3. De repo moet op GitHub staan. Is hij privé? Maak dan onder *Add-ons → Secrets*
   een secret `GITHUB_TOKEN` aan met een fine-grained token met leesrechten.

Draai daarna gewoon alle cellen (of *Save Version → Save & Run All*).

In [ ]:
# --- Parameters ---
REPO_URL = 'https://github.com/JOUW_GEBRUIKERSNAAM/ObjectDetectie.git'

# RD-bbox [xmin, ymin, xmax, ymax] of None om de bbox uit config.yaml te gebruiken.
BBOX = None

# Maximum aantal tegels (None = alles). Zet op bijv. 25 voor een snelle proefrun.
MAX_TEGELS = None

# True: bewaar ook de losse tegels + labels in de output (handig om later
# opnieuw te labelen zonder alles opnieuw te downloaden). Kost extra outputruimte.
HOU_TEGELS = False

In [ ]:
# Repo clonen en afhankelijkheden installeren.
import os, pathlib, subprocess

clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    clone_url = REPO_URL.replace('https://', f'https://{token}@')
    print('GITHUB_TOKEN gevonden — clonen met token (privérepo).')
except Exception:
    print('Geen GITHUB_TOKEN-secret — clonen als publieke repo.')

repo = pathlib.Path('/kaggle/working/ObjectDetectie')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', clone_url, str(repo)], check=True)
%cd {repo}
%pip install -q -r requirements.txt

In [ ]:
# Eventuele bbox-override in config.yaml zetten.
import yaml

cfg_pad = repo / 'config.yaml'
cfg = yaml.safe_load(cfg_pad.read_text(encoding='utf-8'))
if BBOX is not None:
    cfg['gebied']['bbox'] = list(BBOX)
    cfg_pad.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
print('Gebied (RD):', cfg['gebied']['bbox'])
print('Klassen:', cfg['dataset']['klassen'])
DATASET_NAAM = cfg['dataset']['naam']

In [ ]:
# Stap 1-3: BAG ophalen, tegels downloaden, labels genereren.
!python scripts/01_fetch_bag.py
if MAX_TEGELS:
    !python scripts/02_download_tiles.py --max-tegels {MAX_TEGELS}
else:
    !python scripts/02_download_tiles.py
!python scripts/03_make_labels.py

In [ ]:
# Visuele controle: labels over een paar tegels getekend.
from IPython.display import Image as IPyImage, display

!python scripts/05_preview.py --aantal 4
for p in sorted((repo / 'data' / 'preview').glob('*.jpg'))[:4]:
    display(IPyImage(filename=str(p), width=480))

In [ ]:
# Stap 4: dataset bouwen en als notebook-output klaarzetten.
import shutil

!python scripts/04_build_dataset.py --geen-zip

uitvoer = pathlib.Path('/kaggle/working') / DATASET_NAAM
if uitvoer.exists():
    shutil.rmtree(uitvoer)
shutil.move(str(repo / 'data' / 'dataset' / DATASET_NAAM), str(uitvoer))

if HOU_TEGELS:
    tegels_uit = pathlib.Path('/kaggle/working/tiles')
    if tegels_uit.exists():
        shutil.rmtree(tegels_uit)
    shutil.move(str(repo / 'data' / 'tiles'), str(tegels_uit))

# De clone zelf uit de output halen zodat alleen de dataset overblijft.
%cd /kaggle/working
shutil.rmtree(repo)

n_train = len(list((uitvoer / 'images' / 'train').iterdir()))
n_val = len(list((uitvoer / 'images' / 'val').iterdir()))
print(f'Dataset klaar: {uitvoer} ({n_train} train, {n_val} val)')

## Volgende stap: trainen

1. Klik **Save Version → Save & Run All** zodat de output vastligt.
2. Open het trainingsnotebook (`kaggle_train_yolo.ipynb`), kies *Add Input →
   Your Work → Notebooks* en selecteer dit notebook — de dataset wordt dan gemount
   onder `/kaggle/input/…`; het trainingsnotebook vindt de `data.yaml` automatisch.
3. Of plak de trainingscellen onder dit notebook en doe alles in één run
   (zet dan wél een GPU-accelerator aan).

*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*